In [ ]:
!pip install rasterio
!pip install cartopy

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm
import os


rainfall_files = [
    ('Rainfall_2021346N05145.csv','2021346N05145'),
    ('Rainfall_2017347N11131.csv','2017347N11131'),
    ('Rainfall_2014190N08154.csv','2014190N08154')
]

return_levels_df = pd.read_csv('Latest_return_levels_all_periods.csv')
ibtracs_points_path = 'IBTrACS.WP.list.v04r01.points.shp'
ibtracs_lines_path = 'IBTrACS.WP.list.v04r01.lines.shp'

# Path to shapefile
shapefile_path = 'PHL_adm0.shp'


In [ ]:
return_levels_df

,longitude_latitude,return_level_7years,return_level_8years,return_level_9years,return_level_10years,return_level_11years,return_level_12years,return_level_13years,return_level_14years,return_level_15years,...,return_level_991years,return_level_992years,return_level_993years,return_level_994years,return_level_995years,return_level_996years,return_level_997years,return_level_998years,return_level_999years,return_level_1000years
0,123.5505651525227222_8.3500381952534575,142.813948,145.911437,148.643615,151.087633,153.298515,155.316893,157.173620,158.892681,160.493089,...,257.702709,257.726104,257.749476,257.772825,257.796150,257.819451,257.842729,257.865984,257.889216,257.912424
1,125.0505720139454979_7.4500340783997920,90.740508,95.368532,99.450739,103.102397,106.405724,109.421425,112.195601,114.764087,117.155290,...,262.398263,262.433219,262.468140,262.503025,262.537876,262.572691,262.607472,262.642217,262.676928,262.711604
2,119.9505486851080605_15.9500729597955271,558.685309,592.947482,612.311292,623.982047,631.385774,636.282892,639.636470,642.001356,643.711411,...,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986
3,121.5505560039590165_14.0500642686600106,273.830126,279.430504,283.540481,286.663248,289.102543,291.051511,292.638269,293.950824,295.051431,...,305.407226,305.407259,305.407293,305.407326,305.407359,305.407392,305.407425,305.407458,305.407491,305.407524
4,121.3505550891026559_16.0500734172237145,263.884837,273.079488,281.189745,288.444617,295.007447,300.998841,306.510390,311.613287,316.363970,...,604.923047,604.992495,605.061873,605.131181,605.200420,605.269589,605.338688,605.407718,605.476679,605.545571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2414,120.1505495999644211_15.3500702152264168,304.550397,313.271305,320.963691,327.844765,334.069454,339.752150,344.979723,349.819700,354.325610,...,628.017054,628.082924,628.148727,628.214464,628.280135,628.345740,628.411280,628.476753,628.542161,628.607504
2415,125.1505724713736925_7.9500363655407185,159.350744,170.731540,179.316182,185.999698,191.336014,195.685325,199.291439,202.324954,204.908621,...,233.602331,233.602529,233.602726,233.602922,233.603119,233.603314,233.603510,233.603705,233.603899,233.604093
2416,123.4505646950945419_7.6500349932561615,130.319580,135.029249,138.500356,141.147726,143.222730,144.885771,146.243585,147.369699,148.316265,...,157.441865,157.441898,157.441931,157.441964,157.441997,157.442030,157.442063,157.442096,157.442129,157.442161
2417,126.1505770456555382_8.3500381952534575,189.971931,197.274881,203.716538,209.478798,214.691395,219.450124,223.827730,227.880760,231.654041,...,460.845155,460.900315,460.955419,461.010468,461.065461,461.120399,461.175282,461.230110,461.284883,461.339601


In [ ]:
# Parse longitude_latitude column in return_levels_all_periods.csv
# Assuming longitude_latitude is in the format "lon,lat"
return_levels_df[['Longitude', 'Latitude']] = return_levels_df['longitude_latitude'].str.split('_', expand=True)
return_levels_df['Longitude'] = return_levels_df['Longitude'].astype(float)
return_levels_df['Latitude'] = return_levels_df['Latitude'].astype(float)

In [ ]:
# Round coordinates to 11 decimal places for matching
return_levels_df['Longitude'] = return_levels_df['Longitude'].round(11)
return_levels_df['Latitude'] = return_levels_df['Latitude'].round(11)

In [ ]:
return_levels_df

,longitude_latitude,return_level_7years,return_level_8years,return_level_9years,return_level_10years,return_level_11years,return_level_12years,return_level_13years,return_level_14years,return_level_15years,...,return_level_993years,return_level_994years,return_level_995years,return_level_996years,return_level_997years,return_level_998years,return_level_999years,return_level_1000years,Longitude,Latitude
0,123.5505651525227222_8.3500381952534575,142.813948,145.911437,148.643615,151.087633,153.298515,155.316893,157.173620,158.892681,160.493089,...,257.749476,257.772825,257.796150,257.819451,257.842729,257.865984,257.889216,257.912424,123.550565,8.350038
1,125.0505720139454979_7.4500340783997920,90.740508,95.368532,99.450739,103.102397,106.405724,109.421425,112.195601,114.764087,117.155290,...,262.468140,262.503025,262.537876,262.572691,262.607472,262.642217,262.676928,262.711604,125.050572,7.450034
2,119.9505486851080605_15.9500729597955271,558.685309,592.947482,612.311292,623.982047,631.385774,636.282892,639.636470,642.001356,643.711411,...,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,119.950549,15.950073
3,121.5505560039590165_14.0500642686600106,273.830126,279.430504,283.540481,286.663248,289.102543,291.051511,292.638269,293.950824,295.051431,...,305.407293,305.407326,305.407359,305.407392,305.407425,305.407458,305.407491,305.407524,121.550556,14.050064
4,121.3505550891026559_16.0500734172237145,263.884837,273.079488,281.189745,288.444617,295.007447,300.998841,306.510390,311.613287,316.363970,...,605.061873,605.131181,605.200420,605.269589,605.338688,605.407718,605.476679,605.545571,121.350555,16.050073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2414,120.1505495999644211_15.3500702152264168,304.550397,313.271305,320.963691,327.844765,334.069454,339.752150,344.979723,349.819700,354.325610,...,628.148727,628.214464,628.280135,628.345740,628.411280,628.476753,628.542161,628.607504,120.150550,15.350070
2415,125.1505724713736925_7.9500363655407185,159.350744,170.731540,179.316182,185.999698,191.336014,195.685325,199.291439,202.324954,204.908621,...,233.602726,233.602922,233.603119,233.603314,233.603510,233.603705,233.603899,233.604093,125.150572,7.950036
2416,123.4505646950945419_7.6500349932561615,130.319580,135.029249,138.500356,141.147726,143.222730,144.885771,146.243585,147.369699,148.316265,...,157.441931,157.441964,157.441997,157.442030,157.442063,157.442096,157.442129,157.442161,123.450565,7.650035
2417,126.1505770456555382_8.3500381952534575,189.971931,197.274881,203.716538,209.478798,214.691395,219.450124,223.827730,227.880760,231.654041,...,460.955419,461.010468,461.065461,461.120399,461.175282,461.230110,461.284883,461.339601,126.150577,8.350038


In [ ]:
# Define return period mapping to numeric codes
return_period_mapping = {
    'No Data': 0,
    '< 7-year': 1,
    '7-year': 2,
    '8-year': 3,
    '9-year': 4,
    '10-year': 5,
    '11-year': 6,
    '12-year': 7,
    '13-year': 8,
    '14-year': 9,
    '15-year': 10,
    '16-year': 11,
    '17-year': 12,
    '18-year': 13,
    '19-year': 14,
    '20-year': 15,
    '21-year': 16,
    '22-year': 17,
    '23-year': 18,
    '24-year': 19,
    '25-year': 20,
    '26-year': 21,
    '27-year': 22,
    '28-year': 23,
    '29-year': 24,
    '30-year': 25,
    '50-year': 26,
    '100-year': 27,
    '250-year': 28,
    '250-year or greater': 29
}
# Define color map for visualization
cmap = ListedColormap(['gray', 'lightblue'] + [mcolors.LinearSegmentedColormap.from_list('custom', ['lightblue', 'darkred'])(i / 26) for i in range(26)] + ['purple'])
norm = BoundaryNorm(list(range(30)), cmap.N)
color_labels = list(return_period_mapping.keys())

try:
    ibtracs_lines = gpd.read_file(ibtracs_lines_path)
    ibtracs_points = gpd.read_file(ibtracs_points_path)
except Exception as e:
    raise ValueError(f"Error reading IBTrACS files: {e}")


In [ ]:
'''# Function to determine return period
def get_return_period(row):
    rainfall = row['Rainfall']
    if pd.isna(rainfall):
        return 'No Data'
    elif rainfall < row['return_level_25years']:
        return '< 25-year return period'
    elif rainfall < row['return_level_50years']:
        return '25-year'
    elif rainfall < row['return_level_100years']:
        return '50-year'
    elif rainfall < row['return_level_250years']:
        return '100-year'
    elif rainfall < row['return_level_500years']:
        return '250-year'
    else:
        return '250-year or greater'
'''
def get_return_period(row):
    rainfall = row['Rainfall']
    if pd.isna(rainfall):
        return 'No Data'
    elif rainfall < row['return_level_7years']:
        return '< 7-year'
    elif rainfall < row['return_level_8years']:
        return '7-year'
    elif rainfall < row['return_level_9years']:
        return '8-year'
    elif rainfall < row['return_level_10years']:
        return '9-year'
    elif rainfall < row['return_level_11years']:
        return '10-year'
    elif rainfall < row['return_level_12years']:
        return '11-year'
    elif rainfall < row['return_level_13years']:
        return '12-year'
    elif rainfall < row['return_level_14years']:
        return '13-year'
    elif rainfall < row['return_level_15years']:
        return '14-year'
    elif rainfall < row['return_level_16years']:
        return '15-year'
    elif rainfall < row['return_level_17years']:
        return '16-year'
    elif rainfall < row['return_level_18years']:
        return '17-year'
    elif rainfall < row['return_level_19years']:
        return '18-year'
    elif rainfall < row['return_level_20years']:
        return '19-year'
    elif rainfall < row['return_level_21years']:
        return '20-year'
    elif rainfall < row['return_level_22years']:
        return '21-year'
    elif rainfall < row['return_level_23years']:
        return '22-year'
    elif rainfall < row['return_level_24years']:
        return '23-year'
    elif rainfall < row['return_level_25years']:
        return '24-year'
    elif rainfall < row['return_level_26years']:
        return '25-year'
    elif rainfall < row['return_level_27years']:
        return '26-year'
    elif rainfall < row['return_level_28years']:
        return '27-year'
    elif rainfall < row['return_level_29years']:
        return '28-year'
    elif rainfall < row['return_level_30years']:
        return '29-year'
    elif rainfall < row['return_level_50years']:
        return '30-year'
    elif rainfall < row['return_level_100years']:
        return '50-year'
    elif rainfall < row['return_level_250years']:
        return '100-year'
    elif rainfall < row['return_level_500years']:
        return '250-year'
    else:
        return '250-year or greater'


In [ ]:
# Function to create GeoTIFF from CSV
def csv_to_tiff(csv_df, output_tiff, shapefile_path, value_column='Return_Period_Code', resolution=0.1):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Get shapefile bounds
    bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
    lon_min, lat_min, lon_max, lat_max = bounds
    print(f"Shapefile bounds for {output_tiff}: {bounds}")

    # Verify coordinates
    if not (csv_df['Longitude'].between(lon_min, lon_max).any() and csv_df['Latitude'].between(lat_min, lat_max).any()):
        print(f"Warning: No coordinates in CSV fall within shapefile bounds {bounds}")

    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(
        csv_df,
        geometry=[Point(xy) for xy in zip(csv_df['Longitude'], csv_df['Latitude'])],
        crs="EPSG:4326"
    )

    # Calculate grid dimensions
    cols = int((lon_max - lon_min) / resolution) + 1
    rows = int((lat_max - lat_min) / resolution) + 1

    # Create empty grid
    grid = np.full((rows, cols), np.nan, dtype=np.float32)

    # Populate grid with return period codes
    for _, row in csv_df.iterrows():
        col = int((row['Longitude'] - lon_min) / resolution)
        row_idx = int((lat_max - row['Latitude']) / resolution)
        if 0 <= row_idx < rows and 0 <= col < cols:
            grid[row_idx, col] = row[value_column]

    # Define transform for GeoTIFF
    transform = from_origin(lon_min, lat_max, resolution, resolution)

    # Write temporary TIFF
    temp_tiff = 'temp_output.tiff'
    try:
        with rasterio.open(
            temp_tiff,
            'w',
            driver='GTiff',
            height=rows,
            width=cols,
            count=1,
            dtype=grid.dtype,
            crs='EPSG:4326',
            transform=transform,
            nodata=np.nan,
        ) as dst:
            dst.write(grid, 1)
    except Exception as e:
        raise ValueError(f"Error writing temporary TIFF: {e}")

    # Mask the TIFF with shapefile
    try:
        with rasterio.open(temp_tiff) as src:
            masked_data, masked_transform = mask(src, boundary.geometry, crop=True, nodata=np.nan)
        os.remove(temp_tiff)  # Clean up temporary file
    except Exception as e:
        raise ValueError(f"Error masking TIFF with shapefile: {e}")

    # Write final TIFF
    try:
        with rasterio.open(
            output_tiff,
            'w',
            driver='GTiff',
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            count=1,
            dtype=masked_data.dtype,
            crs='EPSG:4326',
            transform=masked_transform,
            nodata=np.nan,
        ) as dst:
            dst.write(masked_data[0], 1)
    except Exception as e:
        raise ValueError(f"Error writing final TIFF: {e}")

# Function to create PNG from GeoTIFF
def tiff_to_png(tiff_file, output_png, shapefile_path,sid):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Read TIFF
    with rasterio.open(tiff_file) as src:
        data = src.read(1)
        transform = src.transform
        extent = [transform.c, transform.c + transform.a * src.width,
                  transform.f + transform.e * src.height, transform.f]

    # Check for valid data
    if np.all(np.isnan(data)):
        print(f"Warning: No valid data in {tiff_file}. Skipping PNG generation.")
        return

    # Create figure with white background
    fig = plt.figure(figsize=(12, 8), facecolor='white')
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_facecolor('white')

    # Set extent to Philippines region
    ax.set_extent([115, 130, 5, 20], crs=ccrs.PlateCarree())

    # Add geographic features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)

    # Plot data with equal aspect for square cells
    cmap.set_bad('white')  # No-data as white
    im = ax.imshow(
        data,
        cmap=cmap,
        norm=norm,
        extent=extent,
        transform=ccrs.PlateCarree(),
        origin='upper',
        aspect='equal'
    )

    # Overlay shapefile boundary
    boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

    # Plot TC track
    track_line = ibtracs_lines[ibtracs_lines['SID'] == sid]
    track_points = ibtracs_points[ibtracs_points['SID'] == sid]
    if not track_line.empty:
        track_line.plot(ax=ax, color='black', linewidth=2, transform=ccrs.PlateCarree())
    if not track_points.empty:
        track_points.plot(ax=ax, color='red', marker='o', markersize=5, linestyle='None', transform=ccrs.PlateCarree())


    # Add gridlines
    ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

    # Add colorbar
    cbar = fig.colorbar(im, ax=ax, ticks=range(len(color_labels)), orientation='vertical')
    cbar.set_label('Return Period')
    cbar.ax.set_yticklabels(color_labels)

    # Set title
    title = f'Return Periods from with TC {sid}'
    ax.set_title(title)

    # Save PNG
    try:
        plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    except Exception as e:
        raise ValueError(f"Error saving PNG: {e}")
    plt.close()

# Process each rainfall CSV
for rainfall_file,sid in rainfall_files:
    # Load the rainfall CSV
    try:
        rainfall_df = pd.read_csv(rainfall_file)
    except Exception as e:
        print(f"Error reading {rainfall_file}: {e}")
        continue

    # Round coordinates for matching
    rainfall_df['Longitude'] = rainfall_df['Longitude'].round(11)
    rainfall_df['Latitude'] = rainfall_df['Latitude'].round(11)

    # Merge with return levels
    merged_df = pd.merge(
        rainfall_df,
        return_levels_df,
        on=['Longitude', 'Latitude'],
        how='right'
    )

    # Determine return periods
    merged_df['Return_Period'] = merged_df.apply(get_return_period, axis=1)

    # Map return periods to numeric codes
    merged_df['Return_Period_Code'] = merged_df['Return_Period'].map(return_period_mapping)

    # Select output columns
    result_df = merged_df[['Longitude', 'Latitude', 'Rainfall', 'Start_time', 'End_time', 'Return_Period']]

    # Save to CSV
    output_csv = f'return_periods_{os.path.basename(rainfall_file)}'

    # Create GeoTIFF
    output_tiff = f'return_periods_{os.path.splitext(os.path.basename(rainfall_file))[0]}.tiff'
    csv_to_tiff(merged_df, output_tiff, shapefile_path, 'Return_Period_Code')

    # Create PNG
    output_png = f'return_periods_{os.path.splitext(os.path.basename(rainfall_file))[0]}.png'
    tiff_to_png(output_tiff, output_png, shapefile_path,sid)

    # Print summary
    print(f"\nProcessed {rainfall_file}:")
    print(f"  CSV saved: {output_csv}")
    print(f"  GeoTIFF saved: {output_tiff}")
    print(f"  PNG saved: {output_png}")
    print(result_df.head())

Shapefile bounds for return_periods_Rainfall_2021346N05145.tiff: [116.94999   5.04917 126.59804  19.39111]

Processed Rainfall_2021346N05145.csv:
  CSV saved: return_periods_Rainfall_2021346N05145.csv
  GeoTIFF saved: return_periods_Rainfall_2021346N05145.tiff
  PNG saved: return_periods_Rainfall_2021346N05145.png
    Longitude   Latitude   Rainfall           Start_time             End_time  \
0  123.550565   8.350038  87.524997  2021-12-16 03:00:00  2021-12-17 15:00:00   
1  125.050572   7.450034  44.379999  2021-12-16 03:00:00  2021-12-17 15:00:00   
2  119.950549  15.950073   8.310000  2021-12-16 03:00:00  2021-12-17 15:00:00   
3  121.550556  14.050064  69.674998  2021-12-16 03:00:00  2021-12-17 15:00:00   
4  121.350555  16.050073  31.724999  2021-12-16 03:00:00  2021-12-17 15:00:00   

  Return_Period  
0      < 7-year  
1      < 7-year  
2      < 7-year  
3      < 7-year  
4      < 7-year  
Shapefile bounds for return_periods_Rainfall_2017347N11131.tiff: [116.94999   5.04917 126

In [ ]:
merged_df

,Longitude,Latitude,Rainfall,Start_time,End_time,longitude_latitude,return_level_7years,return_level_8years,return_level_9years,return_level_10years,...,return_level_993years,return_level_994years,return_level_995years,return_level_996years,return_level_997years,return_level_998years,return_level_999years,return_level_1000years,Return_Period,Return_Period_Code
0,123.550565,8.350038,55.639999,2014-07-14 21:00:00,2014-07-16 06:00:00,123.5505651525227222_8.3500381952534575,142.813948,145.911437,148.643615,151.087633,...,257.749476,257.772825,257.796150,257.819451,257.842729,257.865984,257.889216,257.912424,< 7-year,1
1,125.050572,7.450034,11.505000,2014-07-14 21:00:00,2014-07-16 06:00:00,125.0505720139454979_7.4500340783997920,90.740508,95.368532,99.450739,103.102397,...,262.468140,262.503025,262.537876,262.572691,262.607472,262.642217,262.676928,262.711604,< 7-year,1
2,119.950549,15.950073,26.279999,2014-07-14 21:00:00,2014-07-16 06:00:00,119.9505486851080605_15.9500729597955271,558.685309,592.947482,612.311292,623.982047,...,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,649.919986,< 7-year,1
3,121.550556,14.050064,122.319997,2014-07-14 21:00:00,2014-07-16 06:00:00,121.5505560039590165_14.0500642686600106,273.830126,279.430504,283.540481,286.663248,...,305.407293,305.407326,305.407359,305.407392,305.407425,305.407458,305.407491,305.407524,< 7-year,1
4,121.350555,16.050073,64.949998,2014-07-14 21:00:00,2014-07-16 06:00:00,121.3505550891026559_16.0500734172237145,263.884837,273.079488,281.189745,288.444617,...,605.061873,605.131181,605.200420,605.269589,605.338688,605.407718,605.476679,605.545571,< 7-year,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2414,120.150550,15.350070,107.144997,2014-07-14 21:00:00,2014-07-16 06:00:00,120.1505495999644211_15.3500702152264168,304.550397,313.271305,320.963691,327.844765,...,628.148727,628.214464,628.280135,628.345740,628.411280,628.476753,628.542161,628.607504,< 7-year,1
2415,125.150572,7.950036,9.890000,2014-07-14 21:00:00,2014-07-16 06:00:00,125.1505724713736925_7.9500363655407185,159.350744,170.731540,179.316182,185.999698,...,233.602726,233.602922,233.603119,233.603314,233.603510,233.603705,233.603899,233.604093,< 7-year,1
2416,123.450565,7.650035,21.875000,2014-07-14 21:00:00,2014-07-16 06:00:00,123.4505646950945419_7.6500349932561615,130.319580,135.029249,138.500356,141.147726,...,157.441931,157.441964,157.441997,157.442030,157.442063,157.442096,157.442129,157.442161,< 7-year,1
2417,126.150577,8.350038,11.205000,2014-07-14 21:00:00,2014-07-16 06:00:00,126.1505770456555382_8.3500381952534575,189.971931,197.274881,203.716538,209.478798,...,460.955419,461.010468,461.065461,461.120399,461.175282,461.230110,461.284883,461.339601,< 7-year,1


In [ ]:
    # Check for NaN in longitude_latitude
    nan_count = merged_df['longitude_latitude'].isna().sum()
    print(f"\nNumber of rows with NaN in longitude_latitude for {rainfall_file}: {nan_count}")
    if nan_count > 0:
        print(f"Sample of rows with NaN in longitude_latitude:")
        print(merged_df[merged_df['longitude_latitude'].isna()][['Longitude', 'Latitude', 'Rainfall']].head())



Number of rows with NaN in longitude_latitude for Rainfall_2014190N08154.csv: 0
